# 10 IO编程

IO 就是 Input / Output，即输入和输出。文件读写、在内存中读写数据，都属于 IO

下面的文件操作示例使用 TemporaryDirectory 创建临时目录，退出 with 后会自动清理，可以反复运行。实际使用时，把示例路径换成自己的文件路径即可

## 10.1 文件读写

使用 open() 打开文件，使用 with 可以在结束时自动关闭文件，即使发生异常也会关闭

'r'：读取，文件不存在时会报错

'w'：写入，文件不存在时创建，已存在时会清空原来的内容

'a'：追加，在已有内容末尾写入，文件不存在时创建

'b'：二进制模式，例如 'rb'、'wb'

读写文本时，显式指定 encoding='utf-8'，让编码方式保持一致

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'hello.txt' # 使用 / 拼接路径

    with open(path, 'w', encoding='utf-8') as f:
        f.write('hello, world\n你好，Python\n')

    with open(path, 'a', encoding='utf-8') as f:
        f.write('继续学习文件读写\n')

    with open(path, 'r', encoding='utf-8') as f:
        print(f.read())

读取方法：

read() 一次读取全部内容；read(n) 最多读取 n 个字符，二进制模式下是 n 个字节

readline() 每次读取一行；readlines() 读取全部行并返回 list，通常保留行尾换行符

文件有一个当前位置，读取后会向后移动。seek(0) 可以回到文件开头

大文件适合直接使用 for line in f 逐行读取，避免一次读入全部内容

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'lines.txt'
    with open(path, 'w', encoding='utf-8') as f:
        f.write('第一行\n第二行\n')

    with open(path, 'r', encoding='utf-8') as f:
        print(f.read(3))
        f.seek(0) # 回到开头
        print(f.readline().rstrip('\n'))
        print(f.readlines()) # 从当前位置继续读取
        f.seek(0)
        for line in f:
            print(line.rstrip('\n'))

二进制文件：

图片、音频等通常使用二进制模式读写，操作的是 bytes，而不是 str。二进制模式不能传入 encoding；需要在文本与字节之间转换时，使用 encode()、decode()

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'data.bin'
    with open(path, 'wb') as f:
        f.write('中文'.encode('utf-8'))

    with open(path, 'rb') as f:
        data = f.read()

    print(data)
    print(data.decode('utf-8'))

## 10.2 StringIO 和 BytesIO

StringIO 可以在内存中读写 str，使用方式与文本文件类似

write() 后当前位置在写入内容之后，读取前可以使用 seek(0) 回到开头；getvalue() 可以直接获取全部内容，不改变当前位置

In [ ]:
from io import StringIO

with StringIO() as f:
    f.write('hello')
    f.write('，Python')
    print(f.getvalue())
    f.seek(0)
    print(f.read())

BytesIO 可以在内存中读写 bytes，适合处理二进制数据

In [ ]:
from io import BytesIO

with BytesIO() as f:
    f.write('你好'.encode('utf-8'))
    print(f.getvalue())
    f.seek(0)
    print(f.read().decode('utf-8'))

## 10.3 操作文件和目录

os 和 os.path 提供了常用的文件、目录操作。拼接路径使用 os.path.join()，可以适应不同操作系统的路径分隔符

绝对路径从根目录开始，通常不依赖当前工作目录；相对路径则以当前工作目录为起点

In [ ]:
import os
from tempfile import TemporaryDirectory

print(os.getcwd()) # 当前工作目录

with TemporaryDirectory() as tmpdir:
    folder = os.path.join(tmpdir, 'notes')
    os.mkdir(folder)
    path = os.path.join(folder, 'hello.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('Python学习笔记')

    print(os.path.isdir(folder))
    print(os.path.isfile(path))
    print(os.listdir(folder))
    print(os.path.splitext(path)[1]) # 扩展名

    new_path = os.path.join(folder, 'python.txt')
    os.rename(path, new_path)
    print(os.listdir(folder))
    os.remove(new_path) # 删除示例文件
    os.rmdir(folder) # 删除空目录

pathlib 提供了面向对象的路径操作，Path 对象可以直接传给 open()

常用方法：exists() 检查是否存在，is_file() / is_dir() 判断类型，iterdir() 遍历目录，glob() 按模式查找文件

read_text() / write_text() 可以简化文本文件的读写，它们会自动打开和关闭文件

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    folder = Path(tmpdir) / 'notes'
    folder.mkdir()
    path = folder / 'python.txt'
    path.write_text('学习pathlib', encoding='utf-8')

    print(path.exists())
    print(path.name, path.stem, path.suffix)
    print(path.read_text(encoding='utf-8'))
    print([p.name for p in folder.glob('*.txt')])

## 10.4 序列化

把对象转换成可以保存或传输的数据，这个过程称为序列化；把数据恢复成对象，称为反序列化

pickle 可以把 Python 对象转换为 bytes，适合保存 Python 数据。只反序列化来源可信的 pickle 数据，因为加载时可能执行代码

In [ ]:
import pickle

student = {'name': 'Kan', 'score': 99}
data = pickle.dumps(student) # 对象转换为bytes
print(type(data))
print(pickle.loads(data)) # bytes恢复为对象

pickle.dump() / pickle.load() 用于文件对象，文件需要以二进制模式打开

In [ ]:
import pickle
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'student.pkl'
    student = {'name': 'Kan', 'score': 99}
    with open(path, 'wb') as f:
        pickle.dump(student, f)
    with open(path, 'rb') as f:
        print(pickle.load(f))

JSON：

JSON 是常见的文本数据交换格式，可以在不同语言之间传递数据

json.dumps() 把对象转换为 str，json.loads() 把 JSON 字符串恢复成 Python 对象

ensure_ascii=False 让中文直接显示，indent 可以让输出更易读。JSON 不能直接表示所有 Python 对象，例如 set、自定义类实例；tuple 转换后再读取会变成 list

In [ ]:
import json

student = {'name': '小明', 'score': 99, 'passed': True}
text = json.dumps(student, ensure_ascii=False, indent=2)
print(text)
print(json.loads(text))

json.dump() / json.load() 用于文件对象，保存 JSON 时使用文本模式，并指定 UTF-8 编码

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'student.json'
    student = {'name': '小明', 'score': 99}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(student, f, ensure_ascii=False, indent=2)
    with open(path, 'r', encoding='utf-8') as f:
        restored = json.load(f)
    print(restored)
    print(restored == student)

Practice: 读取一个文本文件，统计非空行的数量，并把统计结果保存到 JSON 文件中

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmpdir:
    folder = Path(tmpdir)
    source_path = folder / 'notes.txt'
    source_path.write_text('Python\n\n   \n文件读写\n', encoding='utf-8')

    count = 0
    with open(source_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip(): # 忽略空行和只有空白字符的行
                count += 1

    result = {'file': source_path.name, 'non_empty_lines': count}
    result_path = folder / 'result.json'
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    with open(result_path, 'r', encoding='utf-8') as f:
        print(json.load(f)) # non_empty_lines为2

参考：[文件读写](https://docs.python.org/3/tutorial/inputoutput.html)、[io](https://docs.python.org/3.11/library/io.html)、[pathlib](https://docs.python.org/3.11/library/pathlib.html)、[pickle](https://docs.python.org/3/library/pickle.html)、[json](https://docs.python.org/3.11/library/json.html)